In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E7.3 · Trust & Safety Evaluation
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Evaluar señales heurísticas sobre las Reviews técnicamente
# MAGIC válidas de Silver.
# MAGIC
# MAGIC Este proceso **NO determina fraude confirmado**.
# MAGIC
# MAGIC Los resultados representan señales potencialmente sospechosas
# MAGIC que pueden ser utilizadas para análisis y priorización.
# MAGIC
# MAGIC ## Entradas
# MAGIC
# MAGIC - `workspace.yelp_silver.slv_review`
# MAGIC - `workspace.yelp_silver.slv_user`
# MAGIC - `workspace.yelp_silver.slv_business`
# MAGIC
# MAGIC ## Salidas
# MAGIC
# MAGIC Gobierno:
# MAGIC - `workspace.yelp_gov.evaluation_rule`
# MAGIC - `workspace.yelp_gov.evaluation_rule_version`
# MAGIC - `workspace.yelp_gov.evaluation_run`
# MAGIC
# MAGIC Evaluación:
# MAGIC - `workspace.yelp_silver.slv_review_evaluation`
# MAGIC - `workspace.yelp_silver.slv_review_signal_summary`
# MAGIC
# MAGIC ## Grano
# MAGIC
# MAGIC `slv_review_evaluation`
# MAGIC
# MAGIC **1 fila = 1 Review × 1 versión de regla × 1 ejecución**
# MAGIC
# MAGIC ## Reglas iniciales
# MAGIC
# MAGIC - R001 · Burst Activity
# MAGIC - R002 · Rating Deviation
# MAGIC - R003 · Repeated Text
# MAGIC - R004 · New Account High Activity
# MAGIC - R005 · User-Business Concentration
# MAGIC - R006 · Business Review Spike


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import uuid

from datetime import (
    datetime,
    timezone
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"

PIPELINE_NAME = "E7_03_trust_safety_evaluacion"
PIPELINE_VERSION = "1.0.0"


# ------------------------------------------------------------
# Desarrollo
#
# True  = prueba con una muestra
# False = ejecución final sobre todo Silver
#
# IMPORTANTE:
# los resultados en DEV_MODE son solo de prueba porque
# algunas métricas de comportamiento se calculan sobre
# una muestra parcial.
# ------------------------------------------------------------

DEV_MODE = False
DEV_LIMIT = 50000


# ------------------------------------------------------------
# Identificador de ejecución
# ------------------------------------------------------------

evaluation_run_id = str(
    uuid.uuid4()
)

evaluation_started_at = (
    datetime
    .now(timezone.utc)
)


print("=" * 72)
print("YELPCONNECT - TRUST & SAFETY EVALUATION")
print("=" * 72)

print(
    f"Evaluation Run : {evaluation_run_id}"
)

print(
    f"Pipeline       : {PIPELINE_NAME}"
)

print(
    f"Version        : {PIPELINE_VERSION}"
)

print(
    f"DEV MODE       : {DEV_MODE}"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # 3. Validaciones previas


# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLAS SILVER
# ============================================================

required_tables = [

    f"{SILVER}.slv_review",
    f"{SILVER}.slv_user",
    f"{SILVER}.slv_business"
]


for table_name in required_tables:

    if not spark.catalog.tableExists(
        table_name
    ):

        raise Exception(
            f"""
            ❌ Tabla requerida no encontrada:

            {table_name}

            Ejecutar primero:
            Bronze → DQ Bronze → Silver → DQ Silver
            """
        )


print(
    "✔ Tablas Silver disponibles."
)


# COMMAND ----------

# ============================================================
# 4. VALIDAR ÚLTIMO QUALITY GATE SILVER
# ============================================================

DQ_RESULT_TABLE = (
    f"{GOV}.dq_resultados"
)


if spark.catalog.tableExists(
    DQ_RESULT_TABLE
):

    ultima_dq_silver = (
        spark.sql(
            f"""
            SELECT
                ejecucion_id

            FROM
                {DQ_RESULT_TABLE}

            WHERE
                capa = 'silver'

            ORDER BY
                ejecucion_ts DESC

            LIMIT 1
            """
        )
        .collect()
    )


    if ultima_dq_silver:

        ultima_dq_id = (
            ultima_dq_silver[0]
            .ejecucion_id
        )


        fallas_criticas = (
            spark.sql(
                f"""
                SELECT
                    COUNT(*) AS cantidad

                FROM
                    {DQ_RESULT_TABLE}

                WHERE
                    ejecucion_id = '{ultima_dq_id}'

                    AND LOWER(
                        criticidad
                    ) = 'alta'

                    AND paso = FALSE
                """
            )
            .first()
            .cantidad
        )


        if fallas_criticas > 0:

            raise Exception(
                f"""
                ❌ DQ Silver posee
                {fallas_criticas}
                fallas críticas.

                Trust & Safety no debe ejecutarse.
                """
            )


        print(
            "✔ Último Quality Gate Silver superado."
        )

    else:

        print(
            "⚠ No se encontró ejecución DQ Silver."
        )

else:

    print(
        "⚠ dq_resultados no disponible."
    )


# COMMAND ----------

# ============================================================
# 5. CREAR / VALIDAR SCHEMA DE GOBIERNO
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOV_SCHEMA}
    """
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 6. Catálogo de reglas Trust & Safety


# COMMAND ----------

# ============================================================
# 6. CREAR evaluation_rule
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {GOV}.evaluation_rule
    (
        rule_id              STRING,
        rule_name            STRING,
        rule_type            STRING,
        business_definition  STRING,
        owner                STRING,
        active_flag          BOOLEAN,
        created_at           TIMESTAMP
    )
    USING DELTA
    """
)


# COMMAND ----------

# ============================================================
# 7. DEFINICIÓN DE REGLAS
# ============================================================

rules = [

    (
        "R001",
        "Burst Activity",
        "behavioral",
        "Detecta concentración elevada de Reviews "
        "del mismo usuario en una ventana corta.",
        "Trust & Safety",
        True
    ),

    (
        "R002",
        "Rating Deviation",
        "behavioral_content",
        "Detecta Reviews cuyo rating presenta una desviación "
        "fuerte respecto del rating agregado del Business.",
        "Trust & Safety",
        True
    ),

    (
        "R003",
        "Repeated Text",
        "content",
        "Detecta textos normalizados repetidos "
        "en múltiples Reviews.",
        "Trust & Safety",
        True
    ),

    (
        "R004",
        "New Account High Activity",
        "behavioral",
        "Detecta actividad elevada durante los primeros "
        "días de vida de una cuenta.",
        "Trust & Safety",
        True
    ),

    (
        "R005",
        "User-Business Concentration",
        "behavioral",
        "Detecta concentración elevada de Reviews "
        "de un usuario hacia un mismo Business.",
        "Trust & Safety",
        True
    ),

    (
        "R006",
        "Business Review Spike",
        "temporal",
        "Detecta incrementos anómalos de Reviews "
        "por Business en un día.",
        "Trust & Safety",
        True
    )
]


df_rules = (
    spark
    .createDataFrame(
        rules,
        """
        rule_id STRING,
        rule_name STRING,
        rule_type STRING,
        business_definition STRING,
        owner STRING,
        active_flag BOOLEAN
        """
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)


df_rules.createOrReplaceTempView(
    "src_evaluation_rule"
)


# COMMAND ----------

# ============================================================
# 8. MERGE evaluation_rule
# ============================================================

spark.sql(
    f"""
    MERGE INTO
        {GOV}.evaluation_rule AS target

    USING
        src_evaluation_rule AS source

    ON
        target.rule_id =
        source.rule_id

    WHEN MATCHED THEN
        UPDATE SET

            target.rule_name =
                source.rule_name,

            target.rule_type =
                source.rule_type,

            target.business_definition =
                source.business_definition,

            target.owner =
                source.owner,

            target.active_flag =
                source.active_flag

    WHEN NOT MATCHED THEN

        INSERT
        (
            rule_id,
            rule_name,
            rule_type,
            business_definition,
            owner,
            active_flag,
            created_at
        )

        VALUES
        (
            source.rule_id,
            source.rule_name,
            source.rule_type,
            source.business_definition,
            source.owner,
            source.active_flag,
            source.created_at
        )
    """
)


print(
    "✔ evaluation_rule actualizado."
)


# COMMAND ----------

# ============================================================
# 9. CREAR evaluation_rule_version
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {GOV}.evaluation_rule_version
    (
        rule_version_id  STRING,
        rule_id          STRING,
        version          STRING,
        description      STRING,
        parameters_json  STRING,
        threshold        DOUBLE,
        weight           DOUBLE,
        valid_from       TIMESTAMP,
        status           STRING
    )
    USING DELTA
    """
)


# COMMAND ----------

# ============================================================
# 10. VERSIONES DE REGLA
# ============================================================

rule_versions = [

    (
        "R001-v1.0",
        "R001",
        "1.0",
        "8 o más Reviews del mismo usuario "
        "durante una ventana de 30 minutos.",
        '{"window_minutes":30,"min_reviews":8}',
        8.0,
        0.20
    ),

    (
        "R002-v1.0",
        "R002",
        "1.0",
        "Diferencia absoluta de al menos 3 estrellas "
        "frente al promedio del Business.",
        '{"rating_difference":3.0}',
        3.0,
        0.15
    ),

    (
        "R003-v1.0",
        "R003",
        "1.0",
        "Texto de al menos 40 caracteres repetido "
        "3 o más veces.",
        '{"min_repetitions":3,"min_text_length":40}',
        3.0,
        0.20
    ),

    (
        "R004-v1.0",
        "R004",
        "1.0",
        "Cuenta con antigüedad máxima de 30 días "
        "y al menos 5 Reviews.",
        '{"account_age_days":30,"min_reviews":5}',
        5.0,
        0.15
    ),

    (
        "R005-v1.0",
        "R005",
        "1.0",
        "Al menos 80% de Reviews del usuario "
        "concentradas en un Business, "
        "con un mínimo de 3 Reviews.",
        '{"concentration":0.80,"min_business_reviews":3}',
        0.80,
        0.15
    ),

    (
        "R006-v1.0",
        "R006",
        "1.0",
        "Al menos 10 Reviews diarias y un volumen "
        "al menos 3 veces superior al promedio diario.",
        '{"min_daily_reviews":10,"multiplier":3.0}',
        3.0,
        0.15
    )
]


df_rule_versions = (
    spark
    .createDataFrame(
        rule_versions,
        """
        rule_version_id STRING,
        rule_id STRING,
        version STRING,
        description STRING,
        parameters_json STRING,
        threshold DOUBLE,
        weight DOUBLE
        """
    )

    .withColumn(
        "valid_from",
        F.current_timestamp()
    )

    .withColumn(
        "status",
        F.lit("ACTIVE")
    )
)


df_rule_versions.createOrReplaceTempView(
    "src_evaluation_rule_version"
)


# COMMAND ----------

# ============================================================
# 11. MERGE VERSIONES
# ============================================================

spark.sql(
    f"""
    MERGE INTO
        {GOV}.evaluation_rule_version AS target

    USING
        src_evaluation_rule_version AS source

    ON
        target.rule_version_id =
        source.rule_version_id

    WHEN MATCHED THEN

        UPDATE SET

            target.description =
                source.description,

            target.parameters_json =
                source.parameters_json,

            target.threshold =
                source.threshold,

            target.weight =
                source.weight,

            target.status =
                source.status

    WHEN NOT MATCHED THEN

        INSERT
        (
            rule_version_id,
            rule_id,
            version,
            description,
            parameters_json,
            threshold,
            weight,
            valid_from,
            status
        )

        VALUES
        (
            source.rule_version_id,
            source.rule_id,
            source.version,
            source.description,
            source.parameters_json,
            source.threshold,
            source.weight,
            source.valid_from,
            source.status
        )
    """
)


print(
    "✔ evaluation_rule_version actualizado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. Registro de ejecución


# COMMAND ----------

# ============================================================
# 12. CREAR evaluation_run
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {GOV}.evaluation_run
    (
        evaluation_run_id  STRING,
        pipeline_name      STRING,
        pipeline_version   STRING,
        dataset_version    STRING,
        started_at         TIMESTAMP,
        finished_at        TIMESTAMP,
        execution_status   STRING
    )
    USING DELTA
    """
)


# COMMAND ----------

# ============================================================
# 13. OBTENER VERSIÓN DATASET SILVER
# ============================================================

dataset_version_row = (
    spark

    .table(
        f"{SILVER}.slv_review"
    )

    .select(
        "_silver_execution_id"
    )

    .where(
        F.col(
            "_silver_execution_id"
        ).isNotNull()
    )

    .limit(1)

    .collect()
)


dataset_version = (
    dataset_version_row[0]
    ._silver_execution_id

    if dataset_version_row

    else "UNKNOWN"
)


print(
    f"Dataset version: "
    f"{dataset_version}"
)


# COMMAND ----------

# ============================================================
# 14. REGISTRAR RUN
# ============================================================

run_schema = """
evaluation_run_id STRING,
pipeline_name STRING,
pipeline_version STRING,
dataset_version STRING,
started_at TIMESTAMP,
finished_at TIMESTAMP,
execution_status STRING
"""


run_df = spark.createDataFrame(

    [
        (
            evaluation_run_id,
            PIPELINE_NAME,
            PIPELINE_VERSION,
            dataset_version,
            evaluation_started_at,
            None,
            "RUNNING"
        )
    ],

    run_schema
)


(
    run_df

    .write

    .format("delta")

    .mode("append")

    .saveAsTable(
        f"{GOV}.evaluation_run"
    )
)


print(
    "✔ Evaluation Run registrado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 15. Preparación del dataset de evaluación


# COMMAND ----------

# ============================================================
# 15. LEER REVIEWS
# ============================================================

reviews = (
    spark.table(
        f"{SILVER}.slv_review"
    )
)


# ------------------------------------------------------------
# DEV MODE
# ------------------------------------------------------------

if DEV_MODE:

    reviews = (
        reviews

        .orderBy(
            "review_id"
        )

        .limit(
            DEV_LIMIT
        )
    )


    print(
        f"⚠ DEV MODE activo: "
        f"{DEV_LIMIT:,} Reviews máximo."
    )


review_count = (
    reviews.count()
)


print(
    f"Reviews a evaluar: "
    f"{review_count:,}"
)


# COMMAND ----------

# ============================================================
# 16. USER FEATURES
# ============================================================

users = (
    spark.table(
        f"{SILVER}.slv_user"
    )

    .select(
        "user_id",
        "yelping_since"
    )
)


# COMMAND ----------

# ============================================================
# 17. BUSINESS FEATURES
# ============================================================

businesses = (
    spark.table(
        f"{SILVER}.slv_business"
    )

    .select(

        "business_id",

        F.col(
            "estrellas"
        ).alias(
            "business_stars"
        )
    )
)


# COMMAND ----------

# ============================================================
# 18. DATASET BASE
# ============================================================

features = (
    reviews

    .join(
        users,
        "user_id",
        "left"
    )

    .join(
        businesses,
        "business_id",
        "left"
    )

    .withColumn(
        "_review_epoch",

        F.col(
            "review_ts"
        ).cast("long")
    )

    .withColumn(
        "_review_date",

        F.to_date(
            "review_ts"
        )
    )

    .withColumn(
        "_normalized_text",

        F.lower(
            F.regexp_replace(
                F.trim(
                    F.col(
                        "review_text"
                    )
                ),
                r"\s+",
                " "
            )
        )
    )

    .withColumn(
        "_text_hash",

        F.sha2(
            F.col(
                "_normalized_text"
            ),
            256
        )
    )

    .withColumn(
        "_text_length",

        F.length(
            "_normalized_text"
        )
    )

    .withColumn(
        "_account_age_days",

        F.datediff(

            F.to_date(
                F.col(
                    "review_ts"
                )
            ),

            F.to_date(
                F.col(
                    "yelping_since"
                )
            )
        )
    )

    .withColumn(
        "_rating_deviation",

        F.abs(
            F.col("stars")
            -
            F.col("business_stars")
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 19. Feature R001 · Burst Activity


# COMMAND ----------

# ============================================================
# 19. R001 - BURST ACTIVITY
# ============================================================

w_burst = (

    Window

    .partitionBy(
        "user_id"
    )

    .orderBy(
        F.col(
            "_review_epoch"
        )
    )

    .rangeBetween(
        -1800,
        0
    )
)


features = (
    features

    .withColumn(
        "_burst_30m_count",

        F.count("*")
        .over(
            w_burst
        )
    )
)


# COMMAND ----------

# ============================================================
# 20. R003 - REPETICIÓN DE TEXTO
# ============================================================

w_text = (
    Window
    .partitionBy(
        "_text_hash"
    )
)


features = (
    features

    .withColumn(
        "_text_repeat_count",

        F.count("*")
        .over(
            w_text
        )
    )
)


# COMMAND ----------

# ============================================================
# 21. R004 / R005 - COMPORTAMIENTO USUARIO
# ============================================================

w_user = (
    Window
    .partitionBy(
        "user_id"
    )
)


w_user_business = (
    Window
    .partitionBy(
        "user_id",
        "business_id"
    )
)


features = (
    features

    .withColumn(
        "_user_review_count",

        F.count("*")
        .over(
            w_user
        )
    )

    .withColumn(
        "_user_business_review_count",

        F.count("*")
        .over(
            w_user_business
        )
    )

    .withColumn(
        "_user_business_concentration",

        F.when(
            F.col(
                "_user_review_count"
            ) > 0,

            F.col(
                "_user_business_review_count"
            )
            /
            F.col(
                "_user_review_count"
            )
        )
    )
)


# COMMAND ----------

# ============================================================
# 22. R006 - BUSINESS REVIEW SPIKE
# ============================================================

business_day = (
    features

    .groupBy(
        "business_id",
        "_review_date"
    )

    .agg(
        F.count("*")
        .alias(
            "_business_day_reviews"
        )
    )
)


business_average = (
    business_day

    .groupBy(
        "business_id"
    )

    .agg(
        F.avg(
            "_business_day_reviews"
        )
        .alias(
            "_business_avg_daily_reviews"
        )
    )
)


business_day = (
    business_day

    .join(
        business_average,
        "business_id",
        "left"
    )
)


features = (
    features

    .join(
        business_day,

        [
            "business_id",
            "_review_date"
        ],

        "left"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 23. Evaluación de reglas


# COMMAND ----------

# ============================================================
# 23. FLAGS DE REGLA
# ============================================================

features = (

    features

    # --------------------------------------------------------
    # R001
    # --------------------------------------------------------

    .withColumn(
        "_r001",

        F.col(
            "_burst_30m_count"
        ) >= 8
    )


    # --------------------------------------------------------
    # R002
    # --------------------------------------------------------

    .withColumn(
        "_r002",

        F.col(
            "_rating_deviation"
        ) >= 3.0
    )


    # --------------------------------------------------------
    # R003
    # --------------------------------------------------------

    .withColumn(
        "_r003",

        (
            F.col(
                "_text_length"
            ) >= 40
        )

        &

        (
            F.col(
                "_text_repeat_count"
            ) >= 3
        )
    )


    # --------------------------------------------------------
    # R004
    # --------------------------------------------------------

    .withColumn(
        "_r004",

        (
            F.col(
                "_account_age_days"
            ).between(
                0,
                30
            )
        )

        &

        (
            F.col(
                "_user_review_count"
            ) >= 5
        )
    )


    # --------------------------------------------------------
    # R005
    # --------------------------------------------------------

    .withColumn(
        "_r005",

        (
            F.col(
                "_user_business_review_count"
            ) >= 3
        )

        &

        (
            F.col(
                "_user_business_concentration"
            ) >= 0.80
        )
    )


    # --------------------------------------------------------
    # R006
    # --------------------------------------------------------

    .withColumn(
        "_r006",

        (
            F.col(
                "_business_day_reviews"
            ) >= 10
        )

        &

        (
            F.col(
                "_business_day_reviews"
            )

            >=

            (
                F.col(
                    "_business_avg_daily_reviews"
                )

                * 3.0
            )
        )
    )
)


# COMMAND ----------

# ============================================================
# 24. FUNCIÓN PARA GENERAR RESULTADO DE REGLA
# ============================================================

def build_rule_evaluation(
    df,
    rule_id,
    rule_version_id,
    flag_column,
    weight,
    reason_column,
    evaluable_column=None
):

    if evaluable_column is None:

        result_expression = (
            F.when(
                F.col(
                    flag_column
                ) == True,
                F.lit("SIGNAL")
            )

            .otherwise(
                F.lit("NO_SIGNAL")
            )
        )

    else:

        result_expression = (
            F.when(
                ~F.col(
                    evaluable_column
                ),
                F.lit(
                    "NOT_EVALUABLE"
                )
            )

            .when(
                F.col(
                    flag_column
                ) == True,
                F.lit(
                    "SIGNAL"
                )
            )

            .otherwise(
                F.lit(
                    "NO_SIGNAL"
                )
            )
        )


    return (

        df

        .select(

            "review_id",

            F.lit(
                rule_id
            ).alias(
                "rule_id"
            ),

            F.lit(
                rule_version_id
            ).alias(
                "rule_version_id"
            ),

            result_expression.alias(
                "evaluation_result"
            ),

            F.when(
                F.col(
                    flag_column
                ) == True,
                F.lit(
                    float(weight)
                )
            )

            .otherwise(
                F.lit(
                    0.0
                )
            )

            .alias(
                "risk_score"
            ),

            reason_column.alias(
                "reason"
            )
        )
    )


# COMMAND ----------

# ============================================================
# 25. R001 RESULT
# ============================================================

r001 = build_rule_evaluation(

    features,

    "R001",

    "R001-v1.0",

    "_r001",

    0.20,

    F.concat(

        F.lit(
            "Reviews del usuario en 30 minutos: "
        ),

        F.col(
            "_burst_30m_count"
        ).cast(
            "string"
        )
    )
)


# COMMAND ----------

# ============================================================
# 26. R002 RESULT
# ============================================================

features = (
    features

    .withColumn(
        "_r002_evaluable",

        F.col(
            "business_stars"
        ).isNotNull()
    )
)


r002 = build_rule_evaluation(

    features,

    "R002",

    "R002-v1.0",

    "_r002",

    0.15,

    F.concat(

        F.lit(
            "Desviación rating: "
        ),

        F.round(
            F.col(
                "_rating_deviation"
            ),
            2
        ).cast(
            "string"
        )
    ),

    "_r002_evaluable"
)


# COMMAND ----------

# ============================================================
# 27. R003 RESULT
# ============================================================

r003 = build_rule_evaluation(

    features,

    "R003",

    "R003-v1.0",

    "_r003",

    0.20,

    F.concat(

        F.lit(
            "Repeticiones del texto normalizado: "
        ),

        F.col(
            "_text_repeat_count"
        ).cast(
            "string"
        )
    )
)


# COMMAND ----------

# ============================================================
# 28. R004 RESULT
# ============================================================

features = (
    features

    .withColumn(
        "_r004_evaluable",

        F.col(
            "yelping_since"
        ).isNotNull()
    )
)


r004 = build_rule_evaluation(

    features,

    "R004",

    "R004-v1.0",

    "_r004",

    0.15,

    F.concat(

        F.lit(
            "Edad cuenta días: "
        ),

        F.col(
            "_account_age_days"
        ).cast(
            "string"
        ),

        F.lit(
            " | Reviews usuario: "
        ),

        F.col(
            "_user_review_count"
        ).cast(
            "string"
        )
    ),

    "_r004_evaluable"
)


# COMMAND ----------

# ============================================================
# 29. R005 RESULT
# ============================================================

r005 = build_rule_evaluation(

    features,

    "R005",

    "R005-v1.0",

    "_r005",

    0.15,

    F.concat(

        F.lit(
            "Reviews hacia Business: "
        ),

        F.col(
            "_user_business_review_count"
        ).cast(
            "string"
        ),

        F.lit(
            " | concentración: "
        ),

        F.round(
            F.col(
                "_user_business_concentration"
            ),
            4
        ).cast(
            "string"
        )
    )
)


# COMMAND ----------

# ============================================================
# 30. R006 RESULT
# ============================================================

r006 = build_rule_evaluation(

    features,

    "R006",

    "R006-v1.0",

    "_r006",

    0.15,

    F.concat(

        F.lit(
            "Reviews día: "
        ),

        F.col(
            "_business_day_reviews"
        ).cast(
            "string"
        ),

        F.lit(
            " | promedio diario: "
        ),

        F.round(
            F.col(
                "_business_avg_daily_reviews"
            ),
            2
        ).cast(
            "string"
        )
    )
)


# COMMAND ----------

# ============================================================
# 31. UNIÓN DE EVALUACIONES
# ============================================================

evaluations = (

    r001

    .unionByName(
        r002
    )

    .unionByName(
        r003
    )

    .unionByName(
        r004
    )

    .unionByName(
        r005
    )

    .unionByName(
        r006
    )
)


# COMMAND ----------

# ============================================================
# 32. METADATA DE EVALUACIÓN
# ============================================================

evaluations = (

    evaluations

    .withColumn(
        "evaluation_run_id",

        F.lit(
            evaluation_run_id
        )
    )

    .withColumn(
        "evaluation_id",

        F.sha2(
            F.concat_ws(
                "||",

                F.col(
                    "review_id"
                ),

                F.col(
                    "rule_version_id"
                ),

                F.col(
                    "evaluation_run_id"
                )
            ),
            256
        )
    )

    .withColumn(
        "evaluated_at",
        F.current_timestamp()
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 33. Persistencia detallada


# COMMAND ----------

# ============================================================
# 33. TABLA DE EVALUACIÓN
# ============================================================

EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)


if not spark.catalog.tableExists(
    EVALUATION_TABLE
):

    (
        evaluations

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(
            EVALUATION_TABLE
        )
    )

else:

    (
        evaluations

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(
            EVALUATION_TABLE
        )
    )


print(
    "✔ slv_review_evaluation persistida."
)


# COMMAND ----------

# ============================================================
# 34. VALIDAR GRANO
# ============================================================

expected_evaluations = (
    review_count
    * 6
)


actual_evaluations = (

    spark

    .table(
        EVALUATION_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        ) == evaluation_run_id
    )

    .count()
)


print(
    f"Evaluaciones esperadas : "
    f"{expected_evaluations:,}"
)

print(
    f"Evaluaciones generadas : "
    f"{actual_evaluations:,}"
)


if (
    expected_evaluations
    != actual_evaluations
):

    raise Exception(
        """
        ❌ El grano esperado
        Review × Regla × Run
        no se cumple.
        """
    )


print(
    "✔ Grano Review × Regla × Run validado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 35. Resumen de señales por Review


# COMMAND ----------

# ============================================================
# 35. RESUMEN REVIEW
# ============================================================

review_signal_summary = (

    evaluations

    .groupBy(
        "review_id",
        "evaluation_run_id"
    )

    .agg(

        F.sum(
            F.when(
                F.col(
                    "evaluation_result"
                ) == "SIGNAL",
                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )

        .alias(
            "signal_count"
        ),


        F.round(
            F.sum(
                "risk_score"
            ),
            4
        )

        .alias(
            "risk_score"
        ),


        F.collect_set(

            F.when(
                F.col(
                    "evaluation_result"
                ) == "SIGNAL",

                F.col(
                    "rule_id"
                )
            )

        ).alias(
            "signal_rules"
        )
    )


    .withColumn(
        "signal_level",

        F.when(
            F.col(
                "risk_score"
            ) >= 0.60,

            F.lit(
                "HIGH"
            )
        )

        .when(
            F.col(
                "risk_score"
            ) >= 0.35,

            F.lit(
                "MEDIUM"
            )
        )

        .when(
            F.col(
                "risk_score"
            ) >= 0.15,

            F.lit(
                "LOW"
            )
        )

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )

    .withColumn(
        "evaluated_at",
        F.current_timestamp()
    )
)


# COMMAND ----------

# ============================================================
# 36. PERSISTIR RESUMEN
# ============================================================

SUMMARY_TABLE = (
    f"{SILVER}.slv_review_signal_summary"
)


if not spark.catalog.tableExists(
    SUMMARY_TABLE
):

    (
        review_signal_summary

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(
            SUMMARY_TABLE
        )
    )

else:

    (
        review_signal_summary

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(
            SUMMARY_TABLE
        )
    )


print(
    "✔ slv_review_signal_summary persistida."
)


# COMMAND ----------

# ============================================================
# 37. FINALIZAR EVALUATION RUN
# ============================================================

evaluation_finished_at = (
    datetime
    .now(timezone.utc)
)


spark.sql(
    f"""
    UPDATE
        {GOV}.evaluation_run

    SET
        finished_at =
            TIMESTAMP(
                '{evaluation_finished_at.isoformat()}'
            ),

        execution_status =
            'SUCCESS'

    WHERE
        evaluation_run_id =
            '{evaluation_run_id}'
    """
)


print(
    "✔ evaluation_run finalizado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 38. Evidencia por regla


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            rule_id,

            evaluation_result,

            COUNT(*) AS reviews

        FROM
            {EVALUATION_TABLE}

        WHERE
            evaluation_run_id =
            '{evaluation_run_id}'

        GROUP BY
            rule_id,
            evaluation_result

        ORDER BY
            rule_id,
            evaluation_result
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 39. Distribución de señal


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            signal_level,

            COUNT(*) AS reviews,

            ROUND(
                COUNT(*) * 100.0
                /
                SUM(
                    COUNT(*)
                ) OVER(),
                2
            ) AS porcentaje

        FROM
            {SUMMARY_TABLE}

        WHERE
            evaluation_run_id =
            '{evaluation_run_id}'

        GROUP BY
            signal_level

        ORDER BY

            CASE signal_level

                WHEN 'HIGH'
                    THEN 1

                WHEN 'MEDIUM'
                    THEN 2

                WHEN 'LOW'
                    THEN 3

                ELSE 4

            END
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 40. Reviews con señales


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            review_id,

            signal_count,

            risk_score,

            signal_level,

            signal_rules

        FROM
            {SUMMARY_TABLE}

        WHERE
            evaluation_run_id =
            '{evaluation_run_id}'

            AND signal_count > 0

        ORDER BY
            risk_score DESC,
            signal_count DESC

        LIMIT 100
        """
    )
)


# COMMAND ----------

# ============================================================
# 41. RESUMEN FINAL
# ============================================================

signal_reviews = (

    review_signal_summary

    .filter(
        F.col(
            "signal_count"
        ) > 0
    )

    .count()
)


high_signal_reviews = (

    review_signal_summary

    .filter(
        F.col(
            "signal_level"
        ) == "HIGH"
    )

    .count()
)


print()
print("=" * 72)
print("TRUST & SAFETY FINALIZADO")
print("=" * 72)

print(
    f"Evaluation Run      : "
    f"{evaluation_run_id}"
)

print(
    f"Reviews evaluadas   : "
    f"{review_count:,}"
)

print(
    f"Reglas              : "
    f"6"
)

print(
    f"Evaluaciones        : "
    f"{actual_evaluations:,}"
)

print(
    f"Reviews con señal   : "
    f"{signal_reviews:,}"
)

print(
    f"Reviews HIGH        : "
    f"{high_signal_reviews:,}"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done — E7.3
# MAGIC
# MAGIC - [ ] `evaluation_rule` creada.
# MAGIC - [ ] `evaluation_rule_version` creada.
# MAGIC - [ ] `evaluation_run` creada.
# MAGIC - [ ] Se ejecutaron 6 reglas.
# MAGIC - [ ] `slv_review_evaluation` creada.
# MAGIC - [ ] Grano Review × Regla × Run validado.
# MAGIC - [ ] `slv_review_signal_summary` creada.
# MAGIC - [ ] Se diferencian SIGNAL / NO_SIGNAL / NOT_EVALUABLE.
# MAGIC - [ ] Se genera `risk_score`.
# MAGIC - [ ] Se genera `signal_level`.
# MAGIC - [ ] No se utiliza etiqueta de fraude confirmado.
# MAGIC - [ ] La ejecución queda trazable mediante `evaluation_run_id`.